# Autocorrelación
### ACF, PACF y el orden de un modelo

`Fase 1 · Video 5` — serie **Forecasting de Ventas de la A a la Z**

En el **Video 4** aislamos los componentes de la serie. Aquí medimos su **memoria**: cuánto predice el
pasado al presente (ACF) y cuánto de forma directa (PACF). Así se leen los órdenes p y q que
alimentarán al ARIMA del Video 13.

---
## 1. Librerías y preparación de datos

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import InterpolationWarning
from pathlib import Path

plt.rcParams.update(
    {
        "figure.facecolor": "#F8FAFC",
        "axes.facecolor": "#F8FAFC",
        "axes.grid": True,
        "grid.color": "#E2E8F0",
        "grid.linewidth": 0.8,
        "font.size": 11,
    }
)
BLUE, RED, GREEN, PURPLE, ORANGE = "#2563EB", "#DC2626", "#16A34A", "#7C3AED", "#EA580C"
money_fmt = FuncFormatter(lambda x, _: f"${x / 1e6:.1f}M")
print("✅ Librerías cargadas")

In [ ]:
def find_csv(filename="sales_history.csv", max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / "output" / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")


csv_path = find_csv()
df_raw = pd.read_csv(csv_path, nrows=2)
date_col = next(c for c in df_raw.columns if "date" in c.strip().lower())
df = pd.read_csv(csv_path, parse_dates=[date_col])
df.rename(columns={date_col: "date"}, inplace=True)
df.sort_values("date", inplace=True)

serie = df.groupby("date")["revenue"].sum().resample("W-MON").sum()
serie.name = "revenue"

print(f"✅ Serie semanal: {len(serie)} observaciones")
print(f"   {serie.index.min().date()} → {serie.index.max().date()}")

In [ ]:
# Preparamos las tres versiones de la serie que usaremos a lo largo del notebook.
# Mantenemos la MISMA receta de los Videos 3 y 4: log (log1p) como estabilizador

serie_log = np.log1p(serie)

# 2) Serie desestacionalizada — STL sobre log1p (igual que el Video 4).
#    Desestacionalización EXACTA: quitamos S en la escala log y volvemos con expm1.
res_stl = STL(serie_log, period=52, seasonal=53, robust=True).fit()
deseason = np.expm1(serie_log - res_stl.seasonal)
deseason.name = "desestacionalizada"

# 3) Serie estacionaria — log deseason + Δ1. Trabajamos en la escala log para que
#    sea también homocedástica (no solo estacionaria en media), consistente con el
#    orden Box-Jenkins del Video 3: primero varianza, luego media.
estacionaria = (serie_log - res_stl.seasonal).diff().dropna()
estacionaria.name = "estacionaria (log deseason + Δ1)"

# Verificamos que de verdad sea estacionaria (no lo damos por hecho)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", InterpolationWarning)
    adf_p = adfuller(estacionaria, autolag="AIC", regression="c")[1]
    kpss_p = kpss(estacionaria, regression="c", nlags="auto")[1]

conf95 = 1.96 / np.sqrt(len(serie))

print("Tres versiones de la serie preparadas:")
print(f"  1. Original           — {len(serie)} obs   (tendencia + estacionalidad)")
print(f"  2. Desestacionalizada — {len(deseason)} obs   (sin ciclos anuales)")
print(f"  3. Estacionaria       — {len(estacionaria)} obs   (log deseason + Δ1)")
print(f"\nVerificación de la serie estacionaria (tests del Video 3):")
print(
    f"  ADF  p={adf_p:.4f}  → {'✅ estacionaria' if adf_p < 0.05 else '❌ no estacionaria'}"
)
print(
    f"  KPSS p={kpss_p:.4f}  → {'✅ estacionaria' if kpss_p >= 0.05 else '❌ no estacionaria'}"
)
print(f"\nIntervalo de confianza 95%: ±{conf95:.4f}")

---
## 2. Intuición: la autocorrelación como un scatter plot

Antes de ver correlogramas, entendamos qué número estamos calculando.

### La fórmula y su traducción

La autocorrelación en el rezago $k$ se define como la covarianza entre la serie y su versión desplazada, normalizada por la varianza:

$$\rho(k) = \frac{\text{Cov}(y_t,\ y_{t-k})}{\sqrt{\text{Var}(y_t)\,\text{Var}(y_{t-k})}}$$

Como es **la misma serie** desplazada, su varianza no cambia ($\text{Var}(y_t) = \text{Var}(y_{t-k})$), así que la fórmula se simplifica a:

$$\rho(k) = \frac{\text{Cov}(y_t,\ y_{t-k})}{\text{Var}(y)}$$

**En cristiano:** $\rho(k)$ es literalmente **una correlación de Pearson** entre dos columnas — la serie original y la misma serie corrida $k$ posiciones. Nada más.

### La analogía del lago

Imagina lanzar una piedra a un lago. La ACF mide **cuánto siguen visibles las ondas a distintas distancias** del punto de impacto:
- Si las ondas desaparecen rápido → la serie tiene **memoria corta**
- Si las ondas persisten lejos → la serie tiene **memoria larga**

### Verlo como scatter plot

Si graficamos $y_t$ contra $y_{t-k}$, la autocorrelación es **qué tan bien se alinean los puntos sobre una recta**:
- Puntos alineados → $\rho(k)$ alta → ese rezago **sí predice** el presente
- Nube dispersa → $\rho(k) \approx 0$ → ese rezago **no aporta** información

| Valor de $\rho(k)$ | Significado |
|---|---|
| $+1$ | Relación lineal positiva perfecta |
| $0$ | No hay relación lineal |
| $-1$ | Relación lineal inversa perfecta |

In [ ]:
lags_demo = [1, 4, 52]  # 1 semana, 1 mes, 1 año
descripciones = {
    1: "una semana atrás",
    4: "un mes atrás",
    52: "el mismo período del año pasado",
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, lag in zip(axes, lags_demo):
    y = serie.values[lag:]
    y_lag = serie.values[:-lag]
    r = np.corrcoef(y, y_lag)[0, 1]

    ax.scatter(y_lag, y, alpha=0.4, color=BLUE, s=18, edgecolor="white", linewidth=0.5)
    m, b = np.polyfit(y_lag, y, 1)
    x_line = np.linspace(y_lag.min(), y_lag.max(), 100)
    ax.plot(x_line, m * x_line + b, color=RED, linewidth=2.5)

    ax.xaxis.set_major_formatter(money_fmt)
    ax.yaxis.set_major_formatter(money_fmt)
    ax.set_title(
        f"Rezago {lag} — {descripciones[lag]}\nr = {r:.3f}",
        fontsize=12,
        fontweight="bold",
    )
    ax.set_xlabel(f"Ventas en t−{lag}")
    ax.set_ylabel("Ventas en t")

fig.suptitle(
    "La autocorrelación es solo una correlación con la serie desplazada",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("Lectura:")
for lag in lags_demo:
    y, y_lag = serie.values[lag:], serie.values[:-lag]
    r = np.corrcoef(y, y_lag)[0, 1]
    fuerza = "fuerte" if abs(r) > 0.7 else "moderada" if abs(r) > 0.4 else "débil"
    print(f"  Rezago {lag:>2}: r = {r:.3f} → correlación {fuerza}")
print(
    "\nEl correlograma ACF junta estos números para TODOS los rezagos en una sola gráfica."
)
print(
    "Nota: acf() usa el estimador estándar (sesgado: divide por n y usa una media global),"
)
print(
    "así que sus valores difieren ligeramente de este r de Pearson calculado por pares."
)

---
## 3. Demostración empírica: un AR(1) que podemos ver

Antes de analizar nuestra serie real, vamos a **construir** un proceso del que conocemos la respuesta de antemano. Esto nos permite calibrar el ojo: ver cómo se *deberían* ver la ACF y la PACF cuando sabemos qué proceso las generó.

Un proceso **AR(1)** (autorregresivo de orden 1) se define así:

$$y_t = \phi\, y_{t-1} + \varepsilon_t$$

El valor de hoy depende **directamente** solo del valor de ayer (más un shock aleatorio $\varepsilon_t$).

### La predicción teórica

Para un AR(1) con $\phi = 0.8$, la teoría dice exactamente qué esperar:

- **ACF:** decae geométricamente — $\rho(1)=0.8,\ \rho(2)=0.8^2=0.64,\ \rho(3)=0.8^3=0.51,\ \dots$  
  El efecto se *propaga*: aunque hoy solo depende de ayer, ayer dependía de anteayer, así que el eco se transmite hacia atrás.
- **PACF:** un único pico en el rezago 1, y **cero** después.  
  Porque la dependencia *directa* termina en el rezago 1 — todo lo demás es propagado.

Vamos a simularlo y comprobar que la teoría se cumple.

In [ ]:
np.random.seed(42)
phi = 0.8

# ArmaProcess: ar=[1, -phi] porque statsmodels usa la convención del polinomio de rezagos
ar1_process = ArmaProcess(ar=[1, -phi], ma=[1])
ar1_serie = pd.Series(ar1_process.generate_sample(nsample=500))

fig = plt.figure(figsize=(14, 9))
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.3)

ax_s = fig.add_subplot(gs[0, :])
ax_s.plot(ar1_serie.values, color=BLUE, linewidth=0.9)
ax_s.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax_s.set_title(
    f"Proceso AR(1) simulado — y(t) = {phi}·y(t−1) + ε(t)",
    fontsize=12,
    fontweight="bold",
)
ax_s.set_xlabel("Tiempo")

ax_acf = fig.add_subplot(gs[1, 0])
ax_pacf = fig.add_subplot(gs[1, 1])
plot_acf(ar1_serie, lags=20, ax=ax_acf, color=BLUE, alpha=0.05)
plot_pacf(ar1_serie, lags=20, ax=ax_pacf, color=PURPLE, alpha=0.05, method="ywm")
ax_acf.set_title(
    "ACF — decae geométricamente (cola larga)", fontsize=11, fontweight="bold"
)
ax_pacf.set_title("PACF — se corta en el rezago 1", fontsize=11, fontweight="bold")

fig.suptitle(
    "Calibrando el ojo: cómo se ve un AR(1) que ya conocemos",
    fontsize=14,
    fontweight="bold",
)
plt.show()

acf_emp = acf(ar1_serie, nlags=6, fft=True)
print("Rezago | ACF teórica (φ^k) | ACF empírica")
print("─" * 46)
for k in range(1, 6):
    print(f"   {k}    |      {phi**k:.4f}       |    {acf_emp[k]:.4f}")
print("\n→ La ACF decae como φ^k. La PACF, en cambio, muere después del rezago 1:")
print("  esa es la firma inconfundible de un AR(1).")

---
## 4. El correlograma ACF: cómo se lee

El correlograma grafica la autocorrelación para **todos los rezagos a la vez**.  
Cada barra es un rezago. La **banda sombreada** es el intervalo de confianza al 95%.

### ¿De dónde sale la banda de confianza?

Bajo la hipótesis nula de *ruido blanco* (la serie no tiene memoria), la autocorrelación muestral se distribuye aproximadamente como:

$$\hat\rho(k) \ \sim\ \mathcal{N}\!\left(0,\ \tfrac{1}{n}\right)$$

De ahí salen las bandas $\pm 1.96/\sqrt{n}$: una barra que las sobrepasa es **estadísticamente significativa** — no es ruido, hay memoria real en ese rezago.

**En cristiano:** la banda es el "nivel del mar". Una barra que sobresale es una ola real; una barra dentro de la banda es solo el chapoteo del azar.

### El catálogo de patrones teóricos

| Proceso | Comportamiento de la ACF | Por qué |
|---|---|---|
| **AR(p)** | Decaimiento exponencial o sinusoidal (cola larga) | $y_t$ depende de rezagos que a su vez dependen de rezagos anteriores → el efecto se propaga |
| **MA(q)** | **Corte abrupto** en $k=q$: $\rho(k)=0$ para todo $k>q$ | Las innovaciones $\varepsilon_{t-j}$ solo aparecen hasta $j=q$ |
| **ARMA(p,q)** | Decaimiento mixto (cola infinita, sin corte exacto) | Combinación de dependencia directa e indirecta |
| **Tendencia** | Decaimiento muy lento, casi recto | La serie no es estacionaria — *no es memoria estructural real* |
| **Estacional** $s$ | Picos en $k = s, 2s, 3s, \dots$ | Dependencia periódica estructural |

Comparamos la serie original (con tendencia) contra la desestacionalizada.

In [ ]:
LAGS = 60

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

plot_acf(serie, lags=LAGS, ax=axes[0], color=BLUE, alpha=0.05)
axes[0].set_title(
    "ACF — Serie Original\n"
    'Decaimiento lento = tendencia. Las barras tardan en "morir".',
    fontsize=12,
    fontweight="bold",
)
axes[0].set_xlabel("Rezago (semanas)")
axes[0].set_ylabel("Autocorrelación")

plot_acf(deseason, lags=LAGS, ax=axes[1], color=GREEN, alpha=0.05)
axes[1].set_title(
    "ACF — Serie Desestacionalizada\n"
    "Sin los picos periódicos: la estacionalidad fue removida en el Video 4.",
    fontsize=12,
    fontweight="bold",
)
axes[1].set_xlabel("Rezago (semanas)")
axes[1].set_ylabel("Autocorrelación")

fig.suptitle("Lectura del correlograma ACF", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

acf_orig = acf(serie, nlags=LAGS, fft=True)
sig_lags = [i for i, v in enumerate(acf_orig) if abs(v) > conf95 and i > 0]
print(f"Serie original: {len(sig_lags)} de {LAGS} rezagos son significativos.")
print(
    "Que tantos rezagos sean significativos es, en sí mismo, una firma de no-estacionariedad:"
)
print(
    "una serie estacionaria tiene pocos rezagos significativos y concentrados al inicio."
)

---
## 5. PACF: por qué necesitamos la correlación *parcial*

La ACF tiene un problema fundamental: **arrastra correlaciones indirectas**.

### El problema, con una cadena causal 🕸️

Imagina una cadena de dependencia:

$$A \rightarrow B \rightarrow C$$

La ACF detecta correlación entre $A$ y $C$ — pero esa correlación es **prestada**: viaja a través de $B$. No existe un vínculo directo entre $A$ y $C$.

Traducido a series de tiempo: si las ventas de hoy dependen de ayer, y las de ayer dependen de anteayer, la ACF en el rezago 2 mostrará correlación. Pero esa correlación viene de la cadena `hoy ← ayer ← anteayer`, no de una relación real entre hoy y anteayer.

### La solución: la PACF

La **PACF** pregunta exactamente lo que necesitamos: *"si ya conozco todos los puntos intermedios, ¿todavía queda relación directa entre estos dos?"*

Formalmente, la PACF en el rezago $k$ es el **último coeficiente** $\phi_{kk}$ de la regresión autorregresiva de orden $k$:

$$y_t = \phi_{k1}\,y_{t-1} + \phi_{k2}\,y_{t-2} + \dots + \phi_{kk}\,y_{t-k} + \varepsilon_t$$

Ese coeficiente $\phi_{kk}$ se obtiene resolviendo el sistema de **ecuaciones de Yule-Walker**, que relaciona los coeficientes con las autocorrelaciones ya conocidas.

**En cristiano:** la PACF hace una regresión que incluye *todos* los rezagos hasta $k$, y se queda solo con la contribución limpia del rezago $k$ — descontando todo lo que ya explicaban los rezagos intermedios.

### El ejemplo numérico que lo demuestra

En el AR(1) con $\phi = 0.8$ que simulamos arriba:
- La **ACF** en el rezago 2 vale $\rho(2) = \phi^2 = 0.64$ — claramente distinta de cero.
- La **PACF** en el rezago 2 vale $\phi_{22} = 0$.

La correlación en el rezago 2 *existe*, pero es **enteramente indirecta** — transmitida vía $y_{t-1}$. La PACF lo detecta y la anula. Esa es toda la magia.

### Resumen de la diferencia

| | ACF | PACF |
|---|---|---|
| Mide | Correlación total (directa + arrastrada) | Correlación directa pura |
| Carácter | Memoria total, más "difusa" | Memoria directa, más "quirúrgica" 🔬 |
| En un AR(p) | Decae gradualmente | **Se corta** en el rezago $p$ |
| En un MA(q) | **Se corta** en el rezago $q$ | Decae gradualmente |

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

plot_pacf(serie, lags=LAGS, ax=axes[0], color=PURPLE, alpha=0.05, method="ywm")
axes[0].set_title(
    "PACF — Serie Original\n"
    "El primer rezago domina: hay fuerte dependencia directa del valor anterior.",
    fontsize=12,
    fontweight="bold",
)
axes[0].set_xlabel("Rezago (semanas)")
axes[0].set_ylabel("Autocorrelación Parcial")

plot_pacf(deseason, lags=LAGS, ax=axes[1], color=ORANGE, alpha=0.05, method="ywm")
axes[1].set_title(
    "PACF — Serie Desestacionalizada\n"
    "Estructura más limpia para identificar el orden AR.",
    fontsize=12,
    fontweight="bold",
)
axes[1].set_xlabel("Rezago (semanas)")
axes[1].set_ylabel("Autocorrelación Parcial")

fig.suptitle(
    "La PACF aísla la correlación directa de cada rezago",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

pacf_orig = pacf(serie, nlags=LAGS, method="ywm")
sig_pacf = [i for i, v in enumerate(pacf_orig) if abs(v) > conf95 and i > 0]
print(f"PACF serie original — rezagos significativos: {sig_pacf[:10]}")
print("\nNota cómo la PACF tiene MENOS rezagos significativos que la ACF:")
print("eso es exactamente lo que esperamos — eliminó las correlaciones arrastradas.")

---
## 6. ACF y PACF sobre la serie estacionaria

Aquí está la regla más importante:

> **Los parámetros p y q se identifican sobre la serie YA estacionaria — nunca sobre la original.**

### Por qué — y por qué importa tanto

Considera una serie con tendencia, por ejemplo una caminata aleatoria $y_t = y_{t-1} + \varepsilon_t$.  
Su ACF será **enorme en muchísimos rezagos** — pero *no porque exista memoria estructural verdadera*, sino simplemente porque la serie no es estacionaria.

Es como intentar escuchar un susurro al lado de una bocina encendida. El decaimiento lento de la ACF (la bocina) *ahoga* cualquier estructura AR/MA real (el susurro). Primero apagamos la bocina — estacionarizamos con las herramientas del Video 3 — y entonces el susurro se vuelve audible.

**Por eso el orden Box-Jenkins es sagrado:** primero verificas estacionariedad (ADF, KPSS, diferenciación), y *después* lees ACF y PACF. Hacerlo al revés produce diagnósticos sin sentido.

Trabajamos sobre la serie **desestacionalizada + diferenciada** que preparamos en la sección 1.

In [ ]:
fig = plt.figure(figsize=(14, 11))
gs = gridspec.GridSpec(2, 2, hspace=0.5, wspace=0.3)

ax_serie = fig.add_subplot(gs[0, :])
ax_serie.plot(
    estacionaria.index, estacionaria.values, color=GREEN, linewidth=1, alpha=0.85
)
ax_serie.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax_serie.set_title(
    "Serie Estacionaria — log deseason + Δ1 (escala log, homocedástica)",
    fontsize=12,
    fontweight="bold",
)

ax_acf = fig.add_subplot(gs[1, 0])
ax_pacf = fig.add_subplot(gs[1, 1])
plot_acf(estacionaria, lags=40, ax=ax_acf, color=BLUE, alpha=0.05)
plot_pacf(estacionaria, lags=40, ax=ax_pacf, color=PURPLE, alpha=0.05, method="ywm")
ax_acf.set_title("ACF → identifica q (orden MA)", fontsize=11, fontweight="bold")
ax_pacf.set_title("PACF → identifica p (orden AR)", fontsize=11, fontweight="bold")
ax_acf.set_xlabel("Rezago")
ax_pacf.set_xlabel("Rezago")

fig.suptitle(
    "Identificación de p y q sobre la serie estacionaria",
    fontsize=14,
    fontweight="bold",
)
plt.show()

---
## 7. La firma estacional en el correlograma

La ACF es excelente detectando ciclos. Si una serie tiene período $s$, la ACF muestra **picos significativos en los múltiplos del período**: $k = s, 2s, 3s, \dots$ La serie literalmente *se reconoce* cada temporada.

Para nuestra serie semanal con ciclo anual, esos picos aparecen en los rezagos **52, 104, 156...**

### Por qué esto importa más allá de lo visual

Esos picos estacionales no son una curiosidad — son lo que te dice que necesitas un modelo **SARIMA** con componente estacional $(P, D, Q)_s$ en lugar de un ARIMA simple. La lógica de identificación es idéntica:
- Picos en la ACF en los múltiplos de $s$ → término estacional $\text{MA}(Q)_s$
- Picos en la PACF en los múltiplos de $s$ → término estacional $\text{AR}(P)_s$

Lo desarrollaremos en el video de SARIMA (Fase 3). Aquí solo aprendemos a *ver* la firma.

Comparamos la ACF de la serie **con** y **sin** estacionalidad para que sea imposible de no ver.

In [ ]:
LAGS_S = 110  # más de dos períodos para ver el armónico en lag 104

acf_con = acf(serie, nlags=LAGS_S, fft=True)
acf_sin = acf(deseason, nlags=LAGS_S, fft=True)
lags_s = np.arange(len(acf_con))

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

for ax, vals, titulo, color in [
    (axes[0], acf_con, "ACF CON estacionalidad — picos rojos en lag 52 y 104", BLUE),
    (axes[1], acf_sin, "ACF SIN estacionalidad — los picos desaparecen", GREEN),
]:
    colores = [RED if (i % 52 == 0 and i > 0) else color for i in range(len(vals))]
    ax.vlines(lags_s, 0, vals, colors=colores, linewidth=1.5, alpha=0.85)
    ax.scatter(lags_s, vals, c=colores, s=14, zorder=3)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(conf95, color=RED, linewidth=1, linestyle="--", alpha=0.5)
    ax.axhline(-conf95, color=RED, linewidth=1, linestyle="--", alpha=0.5)
    for mult in (52, 104):
        ax.axvline(mult, color=RED, linewidth=1.2, linestyle=":", alpha=0.6)
    ax.set_title(titulo, fontsize=12, fontweight="bold")
    ax.set_xlabel("Rezago (semanas)")
    ax.set_ylabel("Autocorrelación")

fig.suptitle("La firma de la estacionalidad en la ACF", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Autocorrelación en los múltiplos del período anual:")
print(f"  Lag  52  — con estacionalidad: {acf_con[52]:+.3f}   sin: {acf_sin[52]:+.3f}")
print(
    f"  Lag 104  — con estacionalidad: {acf_con[104]:+.3f}   sin: {acf_sin[104]:+.3f}"
)
print(
    f"\nLa caída del pico en lag 52 confirma que la desestacionalización del Video 4 funcionó."
)

---
## 8. Ljung-Box vs ACF/PACF: detector de humo vs mapa eléctrico

En el Video 4 usamos el test de **Ljung-Box** para validar el residual. Vale la pena entender exactamente cómo se relaciona con ACF/PACF, porque resuelven problemas distintos.

### El estadístico de Ljung-Box

$$Q = n(n+2) \sum_{k=1}^{h} \frac{\hat\rho(k)^2}{n-k}$$

Bajo $H_0$ (no hay autocorrelación hasta el rezago $h$), $Q$ sigue una distribución $\chi^2$. Es decir: **suma los cuadrados de las autocorrelaciones** y pregunta si el total es demasiado grande para ser azar.

### La diferencia filosófica

| | Ljung-Box | ACF / PACF |
|---|---|---|
| Pregunta que responde | *"¿Hay autocorrelación, sí o no?"* | *"¿En qué rezagos? ¿Con qué intensidad? ¿Con qué forma?"* |
| Tipo de resultado | Binario (un p-valor) | Diagnóstico estructural completo |
| Analogía | 🚨 Detector de humo | ⚡ Mapa eléctrico del edificio |
| Uso típico | Validación final de residuos | Especificación del modelo (elegir p, q) |

**En cristiano:** el detector de humo te avisa que *hay* fuego, pero no dónde. El mapa eléctrico te muestra exactamente qué circuito falla. Ljung-Box te dice *si* hay memoria; ACF/PACF te dicen *dónde* está y *qué forma* tiene.

Lo demostramos sobre la serie estacionaria.

In [ ]:
lb = acorr_ljungbox(estacionaria, lags=[10, 20, 30], return_df=True)

print("─" * 56)
print('  Ljung-Box sobre la serie estacionaria (el "detector de humo")')
print("  H₀: no hay autocorrelación hasta el rezago h")
print("─" * 56)
for lag, row in lb.iterrows():
    hay_memoria = row["lb_pvalue"] < 0.05
    msg = "🔔 detecta memoria" if hay_memoria else "✅ sin memoria detectable"
    print(f"  h={lag:>3}: Q={row['lb_stat']:>9.3f}  p={row['lb_pvalue']:.4f}  → {msg}")
print("─" * 56)
print("  Ljung-Box te dice SI hay memoria. Para saber EN QUÉ rezagos")
print("  y con qué forma, necesitas leer la ACF y la PACF (sección 9).")

---
## 9. Tabla de decisión y advertencias rigurosas

### La regla de oro de Box-Jenkins

| Patrón ACF | Patrón PACF | Modelo sugerido |
|---|---|---|
| Decae exponencialmente | **Se corta** en el rezago $p$ | **AR(p)** → ARIMA(p, d, 0) |
| **Se corta** en el rezago $q$ | Decae exponencialmente | **MA(q)** → ARIMA(0, d, q) |
| Decae exponencialmente | Decae exponencialmente | **ARMA(p,q)** → ARIMA(p, d, q) |
| Sin barras significativas | Sin barras significativas | **Ruido blanco** — ya está modelada |

### ⚠️ Por qué interpretar esto mecánicamente es peligroso

Muchos tutoriales dicen *"PACF corta → AR, ACF corta → MA"* y se quedan ahí. Eso es **peligrosamente simplista**. Tres advertencias rigurosas:

1. **Muestras finitas.** La ACF y PACF *muestrales* son ruidosas. Los "cortes" rara vez son exactos — verás barras pequeñas cruzando la banda por azar. Por eso se prioriza la **parsimonia**: ante la duda, el modelo más simple.

2. **Cercanía a una raíz unitaria.** Un AR(1) con $\phi \approx 0.95$ produce un decaimiento tan lento en la ACF que puede confundirse con no-estacionariedad — o con un MA de orden alto. El contexto (¿ya estacionarizamos?) es indispensable.

3. **No linealidad y quiebres estructurales.** La ACF/PACF solo capturan dependencia **lineal**. Cambios de régimen, volatilidad estocástica o relaciones no lineales no se ven aquí — requieren otras herramientas (GARCH, modelos de umbral).

**La conclusión madura:** los correlogramas **proponen candidatos, no veredictos**. La identificación visual es el punto de partida; la decisión final se toma con criterios de información (AIC, BIC) sobre una rejilla de modelos. Eso lo haremos en el video de ARIMA.

In [ ]:
# Análisis automático: detectamos p y q candidatos sobre la serie estacionaria
NLAGS = 20
acf_v = acf(estacionaria, nlags=NLAGS, fft=True)
pacf_v = pacf(estacionaria, nlags=NLAGS, method="ywm")
conf_e = 1.96 / np.sqrt(len(estacionaria))


def ultimo_lag_significativo(valores, umbral):
    """Devuelve el último rezago significativo antes del primer 'corte'."""
    orden = 0
    for i in range(1, len(valores)):
        if abs(valores[i]) > umbral:
            orden = i
        else:
            break
    return orden


q_cand = ultimo_lag_significativo(acf_v, conf_e)
p_cand = ultimo_lag_significativo(pacf_v, conf_e)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(estacionaria, lags=NLAGS, ax=axes[0], color=BLUE, alpha=0.05)
plot_pacf(estacionaria, lags=NLAGS, ax=axes[1], color=PURPLE, alpha=0.05, method="ywm")
axes[0].set_title(
    f"ACF — corte en lag {q_cand} → q = {q_cand}", fontsize=12, fontweight="bold"
)
axes[1].set_title(
    f"PACF — corte en lag {p_cand} → p = {p_cand}", fontsize=12, fontweight="bold"
)
for ax in axes:
    ax.set_xlabel("Rezago")
fig.suptitle("Identificación automática de p y q", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("═" * 56)
print("  LECTURA DE LOS CORRELOGRAMAS")
print("═" * 56)
print("  Serie analizada : desestacionalizada + Δ1")
print("  d (Video 3)     : 1")
print(f"  q candidato     : {q_cand}   (último lag ACF significativo antes del corte)")
print(f"  p candidato     : {p_cand}   (último lag PACF significativo antes del corte)")
print("─" * 56)
print(f"  Modelo central sugerido : ARIMA({p_cand}, 1, {q_cand})")
print("  Candidatos a comparar con AIC/BIC en la Fase 3:")
print(f"    ARIMA({p_cand}, 1, 0)   — puro autorregresivo")
print(f"    ARIMA(0, 1, {q_cand})   — puro media móvil")
print(f"    ARIMA({p_cand}, 1, {q_cand})   — mixto")
print("─" * 56)
print("  Recordatorio: esto es un PUNTO DE PARTIDA, no un veredicto.")
print("  El AIC/BIC sobre una rejilla decide el modelo final.")
print("═" * 56)

---
## 10. Conclusiones

| Análisis | Hallazgo |
|---|---|
| AR(1) simulado | Confirma la teoría: ACF decae como $\phi^k$, PACF se corta en lag 1 |
| ACF serie original | Decaimiento lento → confirma tendencia (no estacionaria) |
| ACF desestacionalizada | Picos periódicos desaparecen → estacionalidad removida |
| PACF serie original | Primer rezago dominante → fuerte componente AR |
| ACF lag 52 / 104 | Firma estacional clara → sugiere SARIMA |
| Serie estacionaria | Correlograma limpio → p y q identificables |

### Las reglas que te llevas

1. **"La serie tiene memoria"** significa $\rho(k) \neq 0$ para algún $k$. La ACF cuantifica *cuánta* memoria; la PACF aísla cuánta es *directa*.
2. **ACF mide correlación total; PACF mide correlación directa.** Se leen siempre juntas — una confirma a la otra.
3. **Identifica p y q sobre la serie estacionaria, nunca sobre la original.** La tendencia ahoga la señal real.
4. **ACF se corta en q, PACF se corta en p.** La regla de oro de Box-Jenkins.
5. **Picos en los múltiplos del período = estacionalidad** → necesitas SARIMA, no ARIMA.
6. **Los correlogramas proponen candidatos, no veredictos.** Muestras finitas, raíces cercanas a la unidad y no-linealidad pueden engañar. El AIC/BIC decide.

### El flujo completo de preparación para ARIMA

```
Serie original
  ↓  Video 3 — estabilizar varianza (Box-Cox) + diferenciar (d)
  ↓  Video 4 — desestacionalizar (STL)
  ↓  Video 5 — leer ACF (→ q) y PACF (→ p)
  ↓  Fase 3  — ajustar ARIMA(p,d,q) y elegir con AIC/BIC
```

Cuando aprendes a leer ACF y PACF bien, dejas de ver una serie como una lista de números —  
y empiezas a verla como un **sistema dinámico con inercia temporal**.

---

### Próximo video
**Video 6 — Outliers vs. eventos reales: la verdad detrás del dataset**  
Usaremos los residuales de la descomposición STL para detectar semanas anómalas,  
y al final revelaremos cómo se generaron las promociones y picos en el dataset original.